In [47]:
import torch
import torch.nn as nn

In [48]:
class Masked_attention(nn.Module):
    def __init__(self,dim_in,dim_out,dropout,context_length,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_key   = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.dim_key=dim_out 
        self.dropout=nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) 
    def forward(self,x):
        b,num_tokens,d_in=x.shape 
        key = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)
        attention_score = query @ key.transpose(1, 2) 
        attention_score=attention_score.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)         
        scaled_attention_score= torch.softmax(attention_score/torch.sqrt(torch.tensor(self.dim_key)),dim=-1)
        scaled_attention_score=self.dropout(scaled_attention_score)
        context_vector=scaled_attention_score @ value
        return context_vector
           

In [49]:
## Masked attention wrapper
class Multi_head_attention(nn.Module):
    def __init__(self,d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList([Masked_attention(d_in, d_out, dropout, context_length, qkv_bias=False) for _ in range(num_heads)])
    def forward(self,x):
        return torch.concat([head(x) for head in self.heads],dim=-1)    

In [50]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], 
   [0.55, 0.87, 0.66], 
   [0.57, 0.85, 0.64], 
   [0.22, 0.58, 0.33], 
   [0.77, 0.25, 0.10], 
   [0.05, 0.80, 0.55]] 
)  
batch = torch.stack((inputs, inputs), dim=0)

In [51]:
d_in=3
d_out=2
context_length = batch.shape[1]

In [52]:
mha=Multi_head_attention(d_in, d_out, context_length, 0.0, num_heads=2)
out=mha(batch)

#### Implementing actual Masked Multi Head Attention

In [53]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,n_heads,qkv_bias=False):
        super().__init__()
        assert (d_out % n_heads == 0), \
            "d_out must be divisible by num_heads"
        self.d_out=d_out
        ## no of attentio  heads
        self.n_heads=n_heads
        ## dimension of each attention head
        self.head_dim = d_out // n_heads

        self.W_q=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_k=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_v=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout=nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length),
                       diagonal=1))
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        
    def forward(self,x):  
        b,num_token,d_in=x.shape 
        key=self.W_k(x)  # Shape: (b, num_tokens, d_out)
        query=self.W_q(x)
        value=self.W_v(x)

         ### (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        key=key.view(b,num_token,self.n_heads,self.head_dim)
        query=query.view(b,num_token,self.n_heads,self.head_dim)
        value=value.view(b,num_token,self.n_heads,self.head_dim)

        ### (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        key = key.transpose(1, 2)
        query = query.transpose(1, 2)
        value = value.transpose(1, 2)

        attn_scores = query @ key.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_token, :num_token]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
         ## scaled dot product masked attention
        attn_weights = torch.softmax(attn_scores / torch.sqrt(torch.tensor(self.head_dim)), dim=-1)
        attn_weights = self.dropout(attn_weights)

        ### (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        ## same heads are together easy to concatinate
        context_vector= (attn_weights @ value).transpose(1, 2) 

        ## concat  up all heads
        context_vector = context_vector.contiguous().view(b, num_token, self.d_out)
        context_vector=self.out_proj(context_vector)
        return context_vector


In [54]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], 
   [0.55, 0.87, 0.66], 
   [0.57, 0.85, 0.64], 
   [0.22, 0.58, 0.33], 
   [0.77, 0.25, 0.10], 
   [0.05, 0.80, 0.55]] 
)  
batch = torch.stack((inputs, inputs), dim=0)

In [55]:
mah=MultiHeadAttention(d_in=3,d_out=6,context_length=6,dropout=0.1,n_heads=3)
mah(batch)

tensor([[[-0.1496,  0.6158, -0.1626, -0.2495,  0.0041,  0.0948],
         [-0.2787,  0.5915, -0.2719, -0.1308,  0.1732, -0.0954],
         [-0.3327,  0.5945, -0.2991, -0.0849,  0.2171, -0.1653],
         [-0.3597,  0.5333, -0.3049, -0.1103,  0.1913, -0.1881],
         [-0.3295,  0.5284, -0.2987, -0.0989,  0.2095, -0.1610],
         [-0.3424,  0.5400, -0.3107, -0.0786,  0.2227, -0.1836]],

        [[-0.3525,  0.5570, -0.2530, -0.2723,  0.0515, -0.1035],
         [-0.3419,  0.5161, -0.2511, -0.1602,  0.1780, -0.1399],
         [-0.4154,  0.5854, -0.3247, -0.0854,  0.2203, -0.2429],
         [-0.3847,  0.5577, -0.3299, -0.0808,  0.2205, -0.2210],
         [-0.3120,  0.5063, -0.2931, -0.0951,  0.2101, -0.1565],
         [-0.3296,  0.4999, -0.3238, -0.0857,  0.2137, -0.1800]]],
       grad_fn=<ViewBackward0>)